In [ ]:
import jax.numpy as jnp
import liesel.goose as gs
import optax
import tensorflow_probability.substrates.jax.distributions as tfd

import ppptm as gptm

In [ ]:
def opt1():
    # Define the learning rate scheduler
    scheduler = optax.cosine_decay_schedule(init_value=0.001, decay_steps=2000)

    # Define the AdamW optimizer with weight decay
    adamw_optimizer = optax.adamw(learning_rate=scheduler, weight_decay=1e-4)

    gradient_clipping = optax.clip_by_global_norm(1.0)

    # Combine them using optax.chain
    combined_optimizer = optax.chain(gradient_clipping, adamw_optimizer)
    return combined_optimizer

In [ ]:
data = gptm.load_americas()

In [ ]:
locs = gptm.LocationVars.new_from(data.locs, n_subset=64)
y = data.obs[:, locs.locs.ordering]
model = gptm.Model.new_HG(
    y,
    locs,
    g_dist=gptm.G(y, locs, ard=True).new_skewt(),
    coef=gptm.H(
        locs,
        ard=False,
        locwise_amplitude=False,
    ).new_coef(),
)

In [ ]:
result = model.fit(
    stopper=gs.Stopper(max_iter=2_000, patience=30),
    optimizer=opt1(),
    progress_bar=False,
)
position = result.history.position
assert position is not None

In [ ]:
result.plot_loss()

In [ ]:
result.plot_params(subset=[k for k in position if "length_scale" in k])

In [ ]:
result.plot_params(subset=[k for k in position if "loc_mean" in k])

In [ ]:
result.plot_params(subset=[k for k in position if "scale_mean" in k])

In [ ]:
df = gptm.long_df_multiple(
    locs.locs.ordered,
    mu=model.graph.vars["loc"].value,
    sigma=model.graph.vars["scale"].value,
)

In [ ]:
gptm.plot_df(df, "mu")

In [ ]:
gptm.plot_df(df, "sigma")

In [ ]:
model.coef.value.shape

In [ ]:
df = gptm.long_df(locs.locs.ordered, model.spatial_coef.amplitude.value)
gptm.plot_df(df)

In [ ]:
df = gptm.long_df(locs.locs.ordered, model.coef.value.T)
gptm.plot_df(df.query("obs == 20"))

In [ ]:
gptm.plot_df(df.query("obs == 30"))

In [ ]:
from jax.random import key

samp = model.init_dist().sample((500), seed=key(2))
m = samp.mean(axis=0, keepdims=True)
s = samp.std(axis=0, keepdims=True)

In [ ]:
grid = jnp.linspace(-6.0, 6.0, 301)
grid = jnp.expand_dims(grid, -1)
grid = jnp.tile(grid, locs.locs.nloc)

grid2 = s * grid + m
logdet1 = jnp.log(s)
zt, logdet2 = model.init_dist().transformation_and_logdet(grid2)
lp = tfd.Normal(loc=0.0, scale=1.0).log_prob(zt) + logdet1 + logdet2
pdf = jnp.exp(lp)

df = gptm.long_df_multiple(locs.locs.ordered, pdf=pdf, grid=grid, zt=zt, grid2=grid2)

y2 = (y - y.mean(axis=0, keepdims=True)) / y.std(axis=0, keepdims=True)
df2 = gptm.long_df(locs.locs.ordered, y2)

In [ ]:
import plotnine as p9

(
    p9.ggplot(df.query("loc < 5"))
    + p9.geom_line(p9.aes("grid", "pdf"))
    + p9.geom_density(p9.aes("value"), color="red", data=df2.query("loc < 5"))
    + p9.geom_vline(p9.aes(xintercept=0.0))
    + p9.facet_wrap("~loc")
    # + p9.xlim((-5.0, 5.0))
)